# Notebook 8.3 - Traffic Generator

Send a random MNIST request every 2 seconds for 10 minutes to simulate live traffic against `mnist-serving-endpoint`.

In [ ]:
import json
import os
import random
import time
import requests
import numpy as np
import tensorflow as tf

dbutils.widgets.text("endpoint_name", "mnist-serving-endpoint")
dbutils.widgets.text("request_interval_seconds", "2")
dbutils.widgets.text("duration_minutes", "20")

endpoint_name = dbutils.widgets.get("endpoint_name")
request_interval_seconds = int(dbutils.widgets.get("request_interval_seconds"))
duration_minutes = int(dbutils.widgets.get("duration_minutes"))

context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
workspace_url = context.apiUrl().get()
token = context.apiToken().get()
invocations_url = f"{workspace_url}/serving-endpoints/{endpoint_name}/invocations"

(_, _), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_test = x_test.astype("float32") / 255.0
x_test = np.expand_dims(x_test, -1)

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}

print({
    "invocations_url": invocations_url,
    "request_interval_seconds": request_interval_seconds,
    "duration_minutes": duration_minutes,
})

In [ ]:
end_time = time.time() + duration_minutes * 60
sent = 0
errors = 0

while time.time() < end_time:
    index = random.randint(0, len(x_test) - 1)
    sample = x_test[index].reshape(1, 28, 28, 1).tolist()
    label = int(y_test[index])

    payload = {
        "instances": sample
    }

    response = requests.post(invocations_url, headers=headers, data=json.dumps(payload), timeout=30)

    if response.ok:
        sent += 1
        print({
            "sent": sent,
            "label": label,
            "status": response.status_code,
            "response": response.json(),
        })
    else:
        errors += 1
        print({"status": response.status_code, "body": response.text[:200]})

    time.sleep(request_interval_seconds)

print({"total_sent": sent, "errors": errors})